# Qwen3.6-35B-A3B Uncensored GGUF - Chat & API Server Setup

This notebook configures and runs the **HauhauCS/Qwen3.6-35B-A3B-Uncensored** model on Google Colab (Q8_K_P quant).

## 1. Setup & Installations
Mount Google Drive, install `llama-cpp-python` with server and CUDA acceleration, and install `cloudflared` for tunneling.

In [ ]:
!nvidia-smi

Tue Jun 23 04:53:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# 1. Mount Google Drive for model storage
from google.colab import drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Install dependencies (compiled with GPU support and server features)
print("Installing llama-cpp-python server with CUDA support...")
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install "llama-cpp-python[server]" --upgrade --quiet

print("Installing cloudflared for API tunneling...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

print("Setup complete!")

Mounting Google Drive...
Mounted at /content/drive
Installing llama-cpp-python server with CUDA support...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 MB 38.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00
Installing cloudflared for API tunneling...
Setup complete!


## 2. Locate & Load Model
We cache the model on Google Drive to avoid downloading the 50GB file every session. If the file is not there, it will download from Hugging Face.

In [ ]:
import os
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Configure path on Google Drive
drive_cache_dir = "/content/drive/MyDrive/ColabCache/huggingface"
model_filename = "Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf"
model_path = os.path.join(drive_cache_dir, model_filename)

if not os.path.exists(model_path):
    print(f"Model not found at {model_path}. Downloading from Hugging Face...")
    os.makedirs(drive_cache_dir, exist_ok=True)
    model_path = hf_hub_download(
        repo_id="HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive",
        filename=model_filename,
        local_dir=drive_cache_dir,
        local_dir_use_symlinks=False
    )
    print(f"Download complete! Saved to {model_path}")
else:
    print(f"Model found at {model_path}. Skipping download.")

Model not found at /content/drive/MyDrive/ColabCache/huggingface/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf. Downloading from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggr(…):   0%|          | 0.00/43.6G [00:00<?, ?B/s]

Download complete! Saved to /content/drive/MyDrive/ColabCache/huggingface/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf


## 3. Initialize Model

In [ ]:
import gc

# Clean up any existing model in GPU VRAM before loading new one
if 'llm' in globals():
    print("Cleaning up existing LLM instance to free GPU memory...")
    global llm
    del llm
    gc.collect()

print("Loading model into GPU VRAM (this may take a couple of minutes)...")
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,        # Full offload
    n_ctx=32768,            # Or higher (model supports 262k; start conservative on Colab)
    verbose=False,
    # chat_format="chatml" or let it auto-detect
)
print("Model loaded successfully!")

Cleaning up existing LLM instance to free GPU memory...
Loading model into GPU VRAM (this may take a couple of minutes)...


llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Model loaded successfully!


## Option A: Interactive Multiturn Chat Loop
Run this cell to start an interactive multiturn chat session directly in Colab. Type `exit` to quit.

In [ ]:
system_prompt = "You are Assistant Pepe, a helpful and slightly mischievous assistant."
chat_history = []

def format_prompt(history, system):
    prompt_str = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>"
    for role, text in history:
        prompt_str += f"<|start_header_id|>{role}<|end_header_id|>\n\n{text}<|eot_id|>"
    prompt_str += "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return prompt_str

print("Qwen Model: Ready! Ask me anything. (Type 'exit' to quit)\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ['exit', 'quit']:
        print("\nGoodbye!")
        break
    if not user_input.strip():
        continue

    print("\nAssistant: ", end="", flush=True)
    output = llm(
        user_input,  # Let llama.cpp + jinja handle full chat
        max_tokens=1024,
        temperature=0.7,
        top_p=0.9,
        stream=True
    )

    response = ""
    for chunk in output:
        token = chunk["choices"][0]["text"]
        print(token, end="", flush=True)
        response += token
    print("\n")

## Option B: OpenAI-Compatible API Server (for OpenClaw)
Run this cell to start the API server and tunnel script in the background of the persistent Python kernel.

In [ ]:
!pip install starlette-context

In [ ]:
import os
import re
import time

# Clean up old processes
print("Cleaning up old processes...")
!pkill -f llama_cpp
!pkill -f cloudflared
time.sleep(2)

# Launch Cloudflare tunnel
print("Launching Cloudflare tunnel...")
get_ipython().system("cloudflared tunnel --url http://127.0.0.1:8000 > cloudflare.log 2>&1 &")
time.sleep(6)

# Extract public URL
public_url = None
if os.path.exists("cloudflare.log"):
    with open("cloudflare.log", "r") as f:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
        if match:
            public_url = match.group(0)

if public_url:
    print("\n" + "="*80)
    print(f"🚀 PUBLIC API URL FOR OPENCLAW: {public_url}/v1")
    print("="*80 + "\n")
else:
    print("⚠️ Could not detect Cloudflare URL. Check cloudflare.log")

print("Starting llama.cpp server...\n")

# Run server
!python3 -m llama_cpp.server \
    --model {model_path} \
    --n_gpu_layers -1 \
    --host 127.0.0.1 \
    --port 8000 \
    --n_ctx 65536 \
    --jinja \
    --cache-type-k q8_0 \
    --cache-type-v q8_0 \
    --verbose

Cleaning up old processes...
Launching Cloudflare tunnel...


KeyboardInterrupt: 